# Prompt Token Length Quicklook

This notebook samples execution prompts from the trace datasets through the shortcuts datamodule pipeline and
inspects how long they become for a chosen HuggingFace model. Update the configuration cell below to point at
your tokenizer and dataset shards.

In [ ]:
import collections.abc
import typing

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import tqdm
import transformers

import pyine.data.traces.dataset_utils
import pyine.data.utils.splits
import pyine.organisms.datamodules.samples as sample_utils
import pyine.organisms.datamodules.shortcuts as shortcuts
import pyine.organisms.datamodules.shortcuts_configs as shortcuts_configs
import pyine.prompts.manager
import pyine.utils.transformers

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
# ------------ CHANGE THESE SETTINGS IF NEEDED ------------
hf_model_name: str = "Qwen/Qwen3-4B-Instruct-2507"
source_dataset_name: str = "TACO"
trace_dataset_pattern: str = "v1.5/10s10t.*.lmdb"
selected_part_indices: int | list[int] | None = list(range(26))  # set to None to keep every shard
target_subset: str = "train"
max_sample_count: int | None = None
seed: int = 0
render_strategy: typing.Literal["manual", "generation_prompts", "sft_examples"] = "generation_prompts"

# override dataparser settings for the analysis below
dataparser_config_overrides: dict[str, typing.Any] = {
    target_subset: {
        "filtering_config": {
            "max_traces_per_solution": 3,
            "max_code_line_count": 250,
            "max_code_line_length": 250,
            "max_args_length": 500,
        },
        "selection_config": {
            "code_type_prob_map": {
                "original": 1.0,
                "hinted": 0.0,
                "stubbed": 0.0,
                "obfuscated_hinted": 0.0,
                "obfuscated": 0.0,
            },
            "samples_per_family": 1,
            "draw_attempts": 5,
            "fallback_to_orig": True,
        },
        "transform_config": {
            "transform_strategy": "never",
        },
    },
}

# vLLM/GRPO budget settings (keep in sync with `trl vllm-serve` and hf_trainer configs)
VLLM_MAX_MODEL_LEN: int = 8192  # --max_model_len in `trl vllm-serve`
GRPO_MAX_COMPLETION_LEN: int = 3000  # grpo_config.max_completion_length
GRPO_MAX_PROMPT_LEN: int = 5000  # grpo_config.max_prompt_length
max_seq_len_override: int | None = GRPO_MAX_PROMPT_LEN  # use prompt cap for analysis
# ---------------------------------------------------------

In [ ]:
print(f"loading model+tokenizer pair for {hf_model_name!r}")
model = transformers.AutoModelForCausalLM.from_pretrained(hf_model_name)
tokenizer = transformers.AutoTokenizer.from_pretrained(hf_model_name)
model_max_seq_len = pyine.utils.transformers.infer_effective_max_seq_len(model, tokenizer)
print(f"model max sequence length: {model_max_seq_len}")
if max_seq_len_override is not None:
    max_seq_len = min(model_max_seq_len, max_seq_len_override)
else:
    max_seq_len = model_max_seq_len
print(f"target max sequence length for analyses: {max_seq_len}")

# resolve dataset shards
matching_paths = sorted(
    pyine.data.traces.dataset_utils.get_matching_dataset_paths(
        source_dataset_name=source_dataset_name,
        pattern=trace_dataset_pattern,
    )
)
if not matching_paths:
    raise FileNotFoundError(f"no datasets found for name={source_dataset_name!r} pattern={trace_dataset_pattern!r}.")
if selected_part_indices is None:
    dataset_paths = matching_paths
else:
    if isinstance(selected_part_indices, int):
        selected_part_indices = [selected_part_indices]
    invalid = [idx for idx in selected_part_indices if idx < 0 or idx >= len(matching_paths)]
    if invalid:
        raise ValueError(f"invalid shard indices: {invalid}; available shards: 0..{len(matching_paths) - 1}")
    dataset_paths = [matching_paths[idx] for idx in selected_part_indices]
print("using dataset shards:")
for path in dataset_paths:
    print(f"- {path}")

# build the shortcuts datamodule (keeps sample builder logic close to production)
dm_config = shortcuts_configs.get_datamodule_config(
    lmdb_paths=dataset_paths,
    split_file_path=pyine.data.utils.splits.get_dataset_split_file_path(source_dataset_name),
    seed=seed,
    as_pydantic=True,
    dataparser_config_overrides=dataparser_config_overrides,
)
datamodule = shortcuts.ShortcutBiasDataModule(dm_config, verbose=False)
print("preparing datamodule...")
datamodule.prepare_data()
datamodule.setup()
print("datamodule ready")

In [ ]:
if render_strategy == "manual":
    print("rendering prompts manually by applying default code exec template on parser samples...")
    builder = datamodule.get_parser(target_subset)
    if not isinstance(builder, sample_utils.SampleBuilder):
        raise TypeError(f"unexpected parser type: {type(builder)}")
    if len(builder) == 0:
        raise RuntimeError(f"subset {target_subset!r} produced zero samples")
    sample_cap = len(builder) if max_sample_count is None else min(len(builder), max_sample_count)
    print(f"collecting {sample_cap} samples from subset={target_subset!r} (available={len(builder)})")
    samples: list[sample_utils.SampleData] = []
    for idx in tqdm.tqdm(range(sample_cap), desc="sampling prompts"):
        samples.append(builder[idx])
    print("sample collection complete")
    manager = pyine.prompts.manager.get_framework_prompt_manager()
    prompt_template = pyine.prompts.manager.get_prompt_template("code_execution", include_examples=True)
    prompt_rows = []
    for sample in tqdm.tqdm(samples, desc="preparing prompts"):
        prompt_text = prompt_template.format(**sample._asdict())
        prompt_rows.append({"prompt_text": prompt_text})
elif render_strategy == "generation_prompts":
    print("rendering prompts via datamodule huggingface dataset getter")
    prompts_ds = datamodule.get_hf_messages_dataset(
        subset_name=target_subset,
        append_answer=False,
        keep_original_data=True,
        force_regenerate=True,
    )
    prompts_ds = pyine.utils.transformers.prepare_generation_prompts_from_dataset(
        prompts_ds=prompts_ds,
        tokenizer=tokenizer,
        max_seq_len=None,
        keep_extra_fields=True,
        keep_in_memory=datamodule.config.keep_generated_datasets_in_memory,
    )
    prompt_rows = []
    for sample in tqdm.tqdm(prompts_ds, desc="extracting prompt data"):
        assert isinstance(sample, collections.abc.Mapping)
        prompt_rows.append({"prompt_text": sample["text"]})
elif render_strategy == "sft_examples":
    tokenized_dataset = datamodule.get_hf_tokenized_examples_dataset(
        subset_name=target_subset,
        tokenizer=tokenizer,
        model_max_seq_len=None,
        force_regenerate=True,
    )
    collator = pyine.utils.transformers.PaddingCollatorWithPromptMask(
        tokenizer=tokenizer,
        max_length=model_max_seq_len,
    )
    tokenized_loader = torch.utils.data.DataLoader(
        tokenized_dataset,
        batch_size=4,
        shuffle=False,
        num_workers=4,
        collate_fn=collator,
    )
    prompt_rows = []
    for batch in tqdm.tqdm(tokenized_loader, desc="extracting prompt data"):
        assert isinstance(batch, dict)
        assert all(key in batch for key in ["input_ids", "attention_mask", "labels", "input_len"])
        batch_size = batch["input_ids"].shape[0]
        assert all(batch_size == len(batch[k]) for k in ["attention_mask", "labels", "input_len"])
        max_input_length = max(batch["input_len"])
        assert batch["input_ids"].shape[1] == max_input_length
        for batch_idx in range(batch_size):
            prompt_rows.append(
                {
                    "prompt_text": tokenizer.decode(batch["input_ids"][batch_idx]),
                    "token_count": batch["input_len"][batch_idx],
                }
            )
else:
    raise NotImplementedError(f"unknown render strategy: {render_strategy!r}")
prompts_df = pd.DataFrame(prompt_rows)
prompts_df  # noqa: B018 (for display purposes)

In [ ]:
if "token_count" not in prompts_df.columns:
    prompt_lengths = []
    for text in tqdm.tqdm(prompts_df["prompt_text"], desc="tokenizing prompts"):
        tokenized = tokenizer(text, add_special_tokens=False)
        prompt_lengths.append(len(tokenized["input_ids"]))
    prompts_df["token_count"] = prompt_lengths
over_limit_mask = prompts_df["token_count"] > max_seq_len
over_limit_pct = 100 * over_limit_mask.mean()
print(
    f"samples over the token limit ({max_seq_len}):\n\t{over_limit_mask.sum()} / {len(prompts_df)} "
    f"({over_limit_pct:.2f}%); longest prompt = {prompts_df['token_count'].max()} tokens"
)

In [ ]:
# visualize distribution and highlight overflow
if prompts_df.empty:
    raise RuntimeError("no prompts available for plotting")

token_counts = prompts_df["token_count"].to_numpy()
bins = np.histogram_bin_edges(token_counts, bins="doane")
all_counts, _ = np.histogram(token_counts, bins=bins)
over_counts, _ = np.histogram(token_counts[token_counts > max_seq_len], bins=bins)
within_counts = all_counts - over_counts

fig, ax = plt.subplots()
ax.bar(
    bins[:-1],
    within_counts,
    width=np.diff(bins),
    align="edge",
    color="#2a9d8f",
    label="within limit",
)
ax.bar(
    bins[:-1],
    over_counts,
    width=np.diff(bins),
    align="edge",
    bottom=within_counts,
    color="#e76f51",
    label="> max seq len",
)
for i, (bin_start, total_count) in enumerate(zip(bins[:-1], all_counts, strict=False)):
    if total_count > 0:
        ax.text(  # add top-of-bar label indicating the count (if nonzero)
            bin_start + np.diff(bins)[i] / 2,  # x position: center of bar
            total_count,  # y position: top of bar
            f"{int(total_count):,}",  # the count as text
            ha="center",  # horizontal alignment
            va="bottom",  # vertical alignment
            rotation=45,  # 45 degree angle
            fontsize=8,  # adjust size as needed
        )
limit_line = ax.axvline(
    max_seq_len,
    color="#264653",
    linestyle="--",
    linewidth=1.5,
    label=f"max seq len ({max_seq_len})",
)
ax.set_xlabel("Prompt token count")
ax.set_ylabel("Sample count")
ax.set_yscale("log")
ax.set_ylim(bottom=0.5)
ax.set_title(f"{hf_model_name} prompt length distribution (n={len(token_counts)})")
handles, labels = ax.get_legend_handles_labels()
handles.append(limit_line)
labels.append(limit_line.get_label())
ax.legend(handles, labels)
plt.tight_layout()
plt.show()

In [ ]:
# quick summary table
summary = {
    "sample_count": len(prompts_df),
    "mean_tokens": float(prompts_df["token_count"].mean()),
    "p95_tokens": float(prompts_df["token_count"].quantile(0.95)),
    "max_tokens": int(prompts_df["token_count"].max()),
    "over_limit_count": int(over_limit_mask.sum()),
    "over_limit_pct": round(over_limit_pct, 2),
}
pd.Series(summary)

In [ ]:
# vLLM budget analysis for RL training (uses config from cell 2)
prompts_df["headroom_tokens"] = GRPO_MAX_PROMPT_LEN - prompts_df["token_count"]
prompts_df["completion_budget"] = VLLM_MAX_MODEL_LEN - prompts_df["token_count"]

budget_summary = {
    "vllm_max_model_len": VLLM_MAX_MODEL_LEN,
    "grpo_max_prompt_len": GRPO_MAX_PROMPT_LEN,
    "grpo_max_completion_len": GRPO_MAX_COMPLETION_LEN,
    "prompts_over_cap": int((prompts_df["token_count"] > GRPO_MAX_PROMPT_LEN).sum()),
    "prompts_over_cap_pct": round(100 * (prompts_df["token_count"] > GRPO_MAX_PROMPT_LEN).mean(), 2),
    "min_completion_budget": int(prompts_df["completion_budget"].min()),
    "p50_completion_budget": int(prompts_df["completion_budget"].quantile(0.50)),
    "p95_completion_budget": int(prompts_df["completion_budget"].quantile(0.95)),
}
print("vLLM Budget Analysis:")
pd.Series(budget_summary)

In [ ]:
# detailed percentile breakdown
percentiles = [0.50, 0.75, 0.90, 0.95, 0.99, 1.0]
percentile_df = pd.DataFrame(
    {
        "percentile": [f"p{int(p * 100)}" for p in percentiles],
        "token_count": [int(prompts_df["token_count"].quantile(p)) for p in percentiles],
        "over_cap": [prompts_df["token_count"].quantile(p) > GRPO_MAX_PROMPT_LEN for p in percentiles],
    }
)
percentile_df  # noqa: B018 (for display purposes)